In [1]:
pip install numpy==1.23.4

Note: you may need to restart the kernel to use updated packages.


In [2]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Tue Apr 23 12:24:26 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.129.03             Driver Version: 535.129.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla P100-PCIE-16GB           Off | 00000000:00:04.0 Off |                    0 |
| N/A   34C    P0              27W / 250W |      0MiB / 16384MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [3]:
!pip install datasets
!pip install transformers
!pip install huggingface_hub
!pip install torchaudio
!pip install librosa
!pip install jiwer

In [5]:
from kaggle_secrets import UserSecretsClient

def get_huggingface_api_key():
    user_secrets = UserSecretsClient()
    return user_secrets.get_secret("huggingface_api_key")

huggingface_api_key = get_huggingface_api_key()

# Now, use the Hugging Face API key to log in

from huggingface_hub import HfApi, HfFolder

HfFolder.save_token(huggingface_api_key)  # This will save the token to the HfFolder's location

# Verify that the login was successful
api = HfApi()
whoami = api.whoami(token=huggingface_api_key)
print(whoami)

{'type': 'user', 'id': '65d77479f89a0ea5b6642871', 'name': 'Aviral2412', 'fullname': 'Aviral Gandhi', 'email': 'aviralflipkart@gmail.com', 'emailVerified': True, 'canPay': False, 'periodEnd': None, 'isPro': False, 'avatarUrl': '/avatars/1480d2a82608b3081715668cc4a5cd72.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'test', 'role': 'write'}}}


In [6]:
%%capture
!apt install git-lfs

In [148]:
from datasets import load_dataset

data_files = {
    "train": "/kaggle/input/dataset/Training_dataset/Training_dataset.csv",
    "validation": "/kaggle/input/dataset/Validation_dataset/Validation_dataset.csv"
}

dataset = load_dataset('csv', data_files=data_files)


In [149]:
dataset

DatasetDict({
    train: Dataset({
        features: ['path', 'transcript'],
        num_rows: 4662
    })
    validation: Dataset({
        features: ['path', 'transcript'],
        num_rows: 1000
    })
})

In [150]:
# Assuming dataset is already loaded as shown previously

# Function to replace backslashes with forward slashes in the file path
def convert_path(batch):
    batch["path"] = batch["path"].replace("\\", "/")
    return batch

# Apply the function to the train split
dataset["train"] = dataset["train"].map(convert_path, batched=False)
dataset["validation"] = dataset["validation"].map(convert_path, batched=False)

In [ ]:
TARGET_SAMPLING_RATE=16000

In [151]:
import numpy as np
import librosa
import os

def load_and_process_audio(batch, dataset_type):
    directory_path = f"/kaggle/input/dataset/{dataset_type}_dataset"
    full_path = os.path.join(directory_path, batch['path'])
    
    # Load the audio file
    audio, sr = librosa.load(full_path, sr=None)  # Load with original sampling rate
    
    # Resample the audio if necessary
    if sr != TARGET_SAMPLING_RATE:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=TARGET_SAMPLING_RATE)
        sr = TARGET_SAMPLING_RATE
    
    # Normalize audio
    audio = audio / np.max(np.abs(audio))
    
    # Convert audio to a NumPy array
    audio_np = np.array(audio)
    
    # Update the batch directly if possible
    batch['audio'] = audio_np
    batch['sampling_rate'] = sr
    return batch

def apply_audio_loading(dataset):
    # Apply the function to load and process audio for both training and validation datasets
    dataset['train'] = dataset['train'].map(
        lambda batch: load_and_process_audio(batch, 'Training'),
        
        batched=False
    )
    dataset['validation'] = dataset['validation'].map(
        lambda batch: load_and_process_audio(batch, 'Validation'),
    
        batched=False
    )
    return dataset

# Assuming 'dataset' is already loaded
dataset = apply_audio_loading(dataset)

# Check the type of the audio data after the update
print(type(dataset["train"][1]["audio"]))


<class 'list'>


In [152]:
print(type(dataset["train"][1]["audio"]))


<class 'list'>


In [153]:
dataset

DatasetDict({
    train: Dataset({
        features: ['path', 'transcript', 'audio', 'sampling_rate'],
        num_rows: 4662
    })
    validation: Dataset({
        features: ['path', 'transcript', 'audio', 'sampling_rate'],
        num_rows: 1000
    })
})

In [154]:
from datasets import load_dataset, DatasetDict

# Assuming 'dataset' is already loaded and is a DatasetDict
def append_training_path(batch):
    # Append the specified path to the 'path' key
    batch["path"] = "/kaggle/input/dataset/Training_dataset/" + batch["path"]
    return batch

# Apply the function to the train subset
dataset["train"] = dataset["train"].map(append_training_path, batched=False)

# Now, the 'path' in each entry of the train subset has been updated
print(dataset["train"][0]["path"])  # This will print the updated path for the first entry


/kaggle/input/dataset/Training_dataset/train/1249120_13842059_104469105.wav


In [155]:
from datasets import load_dataset, DatasetDict

# Assuming 'dataset' is already loaded and is a DatasetDict
def append_training_path(batch):
    # Append the specified path to the 'path' key
    batch["path"] = "/kaggle/input/dataset/Validation_dataset/" + batch["path"]
    return batch

# Apply the function to the train subset
dataset["validation"] = dataset["validation"].map(append_training_path, batched=False)

# Now, the 'path' in each entry of the train subset has been updated
print(dataset["validation"][0]["path"])  # This will print the updated path for the first entry


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

/kaggle/input/dataset/Validation_dataset/validate/1249120_13842059_105045085.wav


In [156]:
from datasets import ClassLabel
import random
import pandas as pd
from IPython.display import display, HTML

def show_random_elements(dataset, num_examples=10):

    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)

    df = pd.DataFrame(dataset[picks])
    display(HTML(df.to_html()))

In [157]:
show_random_elements(dataset["train"].remove_columns(["audio", "sampling_rate"]), num_examples=10)

,path,transcript
0,/kaggle/input/dataset/Training_dataset/train/1249120_22345789_43469530.wav,I don't do anything at home and still I feel exhausted.
1,/kaggle/input/dataset/Training_dataset/train/1249120_40401830_104932745.wav,The area around my heart doesn't feel good.
2,/kaggle/input/dataset/Training_dataset/train/1249120_44273314_59068162.wav,i have pain like needles in my joints.
3,/kaggle/input/dataset/Training_dataset/train/1249120_39438102_26087116.wav,glass broken and wound my leg badly the wound is so wide
4,/kaggle/input/dataset/Training_dataset/train/1249120_43898158_31555970.wav,I feel emotionally crushed
5,/kaggle/input/dataset/Training_dataset/train/1249120_35444693_54863137.wav,There is a sharp pain in my bicep. I have tried to apply pain relief cream but it does not help.
6,/kaggle/input/dataset/Training_dataset/train/1249120_44140394_44613817.wav,i can't ride my car at night because i have blurry vision.
7,/kaggle/input/dataset/Training_dataset/train/1249120_33251186_88846713.wav,I have a pain in my head
8,/kaggle/input/dataset/Training_dataset/train/1249120_35350617_73411857.wav,My body feels weak and I tried to make it relax but it stills pain me
9,/kaggle/input/dataset/Training_dataset/train/1249120_20518958_83061698.wav,Can't keep focus


In [158]:
dataset

DatasetDict({
    train: Dataset({
        features: ['path', 'transcript', 'audio', 'sampling_rate'],
        num_rows: 4662
    })
    validation: Dataset({
        features: ['path', 'transcript', 'audio', 'sampling_rate'],
        num_rows: 1000
    })
})

In [159]:
import re
chars_to_ignore_regex = '[\,\?\.\!\-\;\:\$\-\,\)\"\&\_\'\(\)\`\+\–\—\“\’\”\"]'

def remove_special_characters(batch):
    batch["transcript"] = re.sub(chars_to_ignore_regex, '', batch["transcript"]).lower() + " "
    return batch

In [160]:
import re

def remove_numbers(batch):
    # Regular expression to match all digits
    digits_regex = '[0-9]'

    # Replace all occurrences of the digits with an empty string
    batch["transcript"] = re.sub(digits_regex, '', batch["transcript"]).lower() + " "
    return batch

In [161]:
dataset = dataset.map(remove_special_characters)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [162]:
dataset = dataset.map(remove_numbers)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [163]:
show_random_elements(dataset["validation"].remove_columns(["audio"]), num_examples=10)

,path,transcript,sampling_rate
0,/kaggle/input/dataset/Validation_dataset/validate/1249120_41336418_50804956.wav,i have a wound between my toes that gets better overnight and then reopens ever day when i wear dress shoes to work,16000
1,/kaggle/input/dataset/Validation_dataset/validate/1249120_35408915_17152178.wav,i feel a sharp pain in my ankle joint when i stand,16000
2,/kaggle/input/dataset/Validation_dataset/validate/1249120_44093303_84595617.wav,i cant sleep i have a hard cough,16000
3,/kaggle/input/dataset/Validation_dataset/validate/1249120_41087148_78155166.wav,my child has cough all night she cant sleep,16000
4,/kaggle/input/dataset/Validation_dataset/validate/1249120_43951421_59477595.wav,i get breakouts on my chest with red patches that get more intense when i get hot,16000
5,/kaggle/input/dataset/Validation_dataset/validate/1249120_43951421_108895312.wav,my skin is itching,16000
6,/kaggle/input/dataset/Validation_dataset/validate/1249120_40725179_90876204.wav,i have a hair shortage,16000
7,/kaggle/input/dataset/Validation_dataset/validate/1249120_35350617_80299909.wav,i was playing basketball yesterday i have sprained ankle,16000
8,/kaggle/input/dataset/Validation_dataset/validate/1249120_29713479_42913440.wav,i notice a lot more hair coming out than usual when i brush my hair,16000
9,/kaggle/input/dataset/Validation_dataset/validate/1249120_6338946_93790761.wav,my knee hurts when i play squash,16000


In [164]:
def extract_all_chars(batch):
  all_text = " ".join(batch["transcript"])
  vocab = list(set(all_text))
  return {"vocab": [vocab], "all_text": [all_text]}

In [165]:
vocabs = dataset.map(extract_all_chars, batched=True, batch_size=-1, keep_in_memory=True, remove_columns=dataset.column_names['train'])


Map:   0%|          | 0/4662 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [166]:
dataset


DatasetDict({
    train: Dataset({
        features: ['path', 'transcript', 'audio', 'sampling_rate'],
        num_rows: 4662
    })
    validation: Dataset({
        features: ['path', 'transcript', 'audio', 'sampling_rate'],
        num_rows: 1000
    })
})

In [167]:
vocab_list = list(set(vocabs["train"]["vocab"][0]) )


In [168]:
vocab_dict = {v: k for k, v in enumerate(vocab_list)}
vocab_dict

{'a': 0,
 'x': 1,
 'd': 2,
 ' ': 3,
 'i': 4,
 'l': 5,
 'b': 6,
 'k': 7,
 'w': 8,
 'j': 9,
 'c': 10,
 'ï': 11,
 '√': 12,
 'g': 13,
 'm': 14,
 'z': 15,
 'p': 16,
 'n': 17,
 'e': 18,
 'h': 19,
 'f': 20,
 'o': 21,
 's': 22,
 't': 23,
 'q': 24,
 'v': 25,
 'y': 26,
 'u': 27,
 'r': 28}

In [169]:
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]

In [170]:
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)
len(vocab_dict)

31

In [171]:
import json
with open('vocab.json', 'w') as vocab_file:
    json.dump(vocab_dict, vocab_file)

In [173]:
from transformers import Wav2Vec2CTCTokenizer

tokenizer = Wav2Vec2CTCTokenizer("./vocab.json", unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|")

In [174]:
repo_name = "Aviral2412/fineturning-medical_data"

In [175]:
import huggingface_hub
tokenizer.push_to_hub(repo_name)

CommitInfo(commit_url='https://huggingface.co/Aviral2412/fineturning-medical_data/commit/e4d7f632173ffc7f3e88c044563aafa56bddf8a3', commit_message='Upload tokenizer', commit_description='', oid='e4d7f632173ffc7f3e88c044563aafa56bddf8a3', pr_url=None, pr_revision=None, pr_num=None)

In [176]:
from transformers import Wav2Vec2FeatureExtractor

feature_extractor = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16000, padding_value=0.0, do_normalize=True, return_attention_mask=False)

In [177]:
from transformers import Wav2Vec2Processor

processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

In [ ]:
!pip install torchaudio

In [ ]:
!sudo apt update
!sudo apt install ffmpeg


In [178]:
import IPython.display as ipd
import numpy as np
import random

# Assuming 'dataset' is already loaded and processed as shown previously
rand_int = random.randint(0, len(dataset["train"]) - 1)

print(dataset["train"][rand_int]["transcript"])
# Convert the list of audio samples to a numpy array and play it
ipd.Audio(data=np.asarray(dataset["train"][rand_int]["audio"]), autoplay=True, rate=16000)


i have a pain in my muscle  


In [179]:
dataset

DatasetDict({
    train: Dataset({
        features: ['path', 'transcript', 'audio', 'sampling_rate'],
        num_rows: 4662
    })
    validation: Dataset({
        features: ['path', 'transcript', 'audio', 'sampling_rate'],
        num_rows: 1000
    })
})

In [180]:
import librosa
import numpy as np
import os

def prepare_dataset(batch):
    file_path = batch["path"]
    # Initialize placeholders for the expected keys with consistent data types
    placeholders = {
        "input_values": np.array([], dtype=np.float32),  # Ensure consistent dtype
        "input_length": 0,
        "labels": np.array([], dtype=np.int32),  # Ensure consistent dtype
    }

    if not os.path.exists(file_path):
        print(f"The file {file_path} does not exist. Skipping...")
        return placeholders  # Return placeholders to skip this file

    try:
        # Load and resample the audio file at 16 kHz
        waveform, sample_rate = librosa.load(file_path, sr=16000)
    except Exception as e:
        print(f"Error loading {file_path}: {e}. Skipping...")
        return placeholders  # Return placeholders to skip this file

    # Process audio
    input_values = processor(waveform, sampling_rate=sample_rate, return_tensors="np").input_values[0]
    batch["input_values"] = input_values.astype(np.float32)  # Ensure consistent dtype
    batch["input_length"] = len(input_values)

    # Process labels (assuming 'transcript' is the key for text data in your batch)
    if "transcript" in batch:
        # Use the tokenizer to encode the text data
        labels = tokenizer(batch["transcript"]).input_ids
        batch["labels"] = np.array(labels, dtype=np.int32)  # Ensure consistent dtype
    else:
        print(f"No transcript available for {file_path}. Skipping label processing...")
        batch["labels"] = np.array([], dtype=np.int32)

    return batch


In [181]:
columns_to_remove = ['path', 'audio','transcript']  # Specify the columns you want to remove
dataset = dataset.map(prepare_dataset, remove_columns=columns_to_remove)


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [184]:
dataset = dataset.remove_columns(["sampling_rate"])

In [185]:
dataset

DatasetDict({
    train: Dataset({
        features: ['input_values', 'input_length', 'labels'],
        num_rows: 3139
    })
    validation: Dataset({
        features: ['input_values', 'input_length', 'labels'],
        num_rows: 668
    })
})

In [186]:
max_input_length_in_sec = 5.0
dataset["train"] = dataset["train"].filter(lambda x: x < max_input_length_in_sec * processor.feature_extractor.sampling_rate, input_columns=["input_length"])
dataset["validation"] = dataset["validation"].filter(lambda x: x < max_input_length_in_sec * processor.feature_extractor.sampling_rate, input_columns=["input_length"])

Filter:   0%|          | 0/3139 [00:00<?, ? examples/s]

Filter:   0%|          | 0/668 [00:00<?, ? examples/s]

In [187]:
dataset

DatasetDict({
    train: Dataset({
        features: ['input_values', 'input_length', 'labels'],
        num_rows: 3139
    })
    validation: Dataset({
        features: ['input_values', 'input_length', 'labels'],
        num_rows: 668
    })
})

In [188]:
import torch

from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union

@dataclass
class DataCollatorCTCWithPadding:
    """
    Data collator that will dynamically pad the inputs received.
    Args:
        processor (:class:`~transformers.Wav2Vec2Processor`)
            The processor used for proccessing the data.
        padding (:obj:`bool`, :obj:`str` or :class:`~transformers.tokenization_utils_base.PaddingStrategy`, `optional`, defaults to :obj:`True`):
            Select a strategy to pad the returned sequences (according to the model's padding side and padding index)
            among:
            * :obj:`True` or :obj:`'longest'`: Pad to the longest sequence in the batch (or no padding if only a single
              sequence if provided).
            * :obj:`'max_length'`: Pad to a maximum length specified with the argument :obj:`max_length` or to the
              maximum acceptable input length for the model if that argument is not provided.
            * :obj:`False` or :obj:`'do_not_pad'` (default): No padding (i.e., can output a batch with sequences of
              different lengths).
    """

    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lenghts and need
        # different padding methods
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )
        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(
                label_features,
                padding=self.padding,
                return_tensors="pt",
            )

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        batch["labels"] = labels

        return batch

In [189]:
data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

In [190]:
from datasets import load_metric
wer_metric = load_metric("wer")

/tmp/ipykernel_1150/2912728219.py:2: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  wer_metric = load_metric("wer")
/opt/conda/lib/python3.10/site-packages/datasets/load.py:756: FutureWarning: The repository for wer contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.18.0/metrics/wer/wer.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(


In [191]:
def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    # we do not want to group tokens when computing the metrics
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

In [193]:
pip install --upgrade safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.2 MB/s eta 0:00:0000:01
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.4.2
    Uninstalling safetensors-0.4.2:
      Successfully uninstalled safetensors-0.4.2
Note: you may need to restart the kernel to use updated packages.


In [192]:
from transformers import Wav2Vec2ForCTC

model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-base",
    ctc_loss_reduction="mean", 
    pad_token_id=processor.tokenizer.pad_token_id,
)

config.json:   0%|          | 0.00/1.84k [00:00<?, ?B/s]

/opt/conda/lib/python3.10/site-packages/transformers/configuration_utils.py:363: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

/opt/conda/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['lm_head.bias', 'lm_head.weight', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [194]:
model.freeze_feature_encoder()

In [195]:
!pip install accelerate -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.6/297.6 kB 7.6 MB/s eta 0:00:00:00:01
  Attempting uninstall: accelerate
    Found existing installation: accelerate 0.28.0
    Uninstalling accelerate-0.28.0:
      Successfully uninstalled accelerate-0.28.0


In [196]:
!pip install transformers[torch]

In [197]:
!pip install numpy -U

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 70.2 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.23.4
    Uninstalling numpy-1.23.4:
      Successfully uninstalled numpy-1.23.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf 23.8.0 requires cubinlinker, which is not installed.
cudf 23.8.0 requires cupy-cuda11x>=12.0.0, which is not installed.
cudf 23.8.0 requires ptxcompiler, which is not installed.
cuml 23.8.0 requires cupy-cuda11x>=12.0.0, which is not installed.
dask-cudf 23.8.0 requires cupy-cuda11x>=12.0.0, which is not installed.
keras-nlp 0.8.2 requires keras-core, which is not installed.
tensorflow-decision-forests 1.8.1 requires wurlitzer, which is not installed.
apache-beam 2.46.0 requires dill<0.3.2,>=0.

In [198]:
!pip show transformers
!pip show accelerate

Name: transformers
Version: 4.39.3
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /opt/conda/lib/python3.10/site-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: kaggle-environments
Name: accelerate
Version: 0.29.3
Summary: Accelerate
Home-page: https://github.com/huggingface/accelerate
Author: The HuggingFace team
Author-email: zach.mueller@huggingface.co
License: Apache
Location: /opt/conda/lib/python3.10/site-packages
Requires: huggingface-hub, numpy, packaging, psutil, pyyaml, safetensors, torch
Required-by: catalyst


In [199]:
!pip install --upgrade filelock huggingface-hub numpy packaging pyyaml regex requests safetensors tokenizers tqdm torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.5/53.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 774.0/774.0 kB 20.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 63.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 82.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 70.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 961.3 kB/s eta

In [200]:
!pip install transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 42.2 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1


In [201]:
dataset = dataset.filter(lambda example: len(example['labels']) > 0)


Filter:   0%|          | 0/3139 [00:00<?, ? examples/s]

Filter:   0%|          | 0/668 [00:00<?, ? examples/s]

In [202]:
empty_labels = [example for example in dataset if isinstance(example, list) and any(isinstance(label, str) and not label.strip() for label in example)]


In [203]:
dataset


DatasetDict({
    train: Dataset({
        features: ['input_values', 'input_length', 'labels'],
        num_rows: 3139
    })
    validation: Dataset({
        features: ['input_values', 'input_length', 'labels'],
        num_rows: 668
    })
})

In [204]:
from transformers import TrainingArguments
import accelerate
import transformers

training_args = TrainingArguments(
  output_dir=repo_name,
  group_by_length=True,
  per_device_train_batch_size=16,
  gradient_accumulation_steps=2,
  evaluation_strategy="steps",
  num_train_epochs=35,  # Increase epochs
  fp16=True,
  gradient_checkpointing=True,
  save_steps=500,
  eval_steps=500,
  logging_steps=500,
  learning_rate=3e-4,  # Adjust learning rate
  weight_decay=0.01,  # Adjust weight decay
  warmup_steps=500,  # Adjust warmup steps
  save_total_limit=2,
  lr_scheduler_type='linear',  # Use a learning rate scheduler
)


In [206]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=processor.feature_extractor,
)

/opt/conda/lib/python3.10/site-packages/accelerate/accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  deprecated_dl_args["use_seedable_sampler"] = use_seedable_sampler


In [207]:
dataset

DatasetDict({
    train: Dataset({
        features: ['input_values', 'input_length', 'labels'],
        num_rows: 3139
    })
    validation: Dataset({
        features: ['input_values', 'input_length', 'labels'],
        num_rows: 668
    })
})

In [208]:
from kaggle_secrets import UserSecretsClient
import wandb

user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("wandb_api")

wandb.login(key=api_key)

wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [209]:
wandb.init()

wandb: Currently logged in as: aviral241202 (iiit_mini_project). Use `wandb login --relogin` to force relogin


In [210]:
trainer.train()

/opt/conda/lib/python3.10/site-packages/transformers/models/wav2vec2/processing_wav2vec2.py:156: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  use_reentrant(bool, optional): Use checkpointing


Step,Training Loss,Validation Loss,Wer
500,2.543200,0.565681,0.374223
1000,0.369100,0.406448,0.266465
1500,0.204500,0.450041,0.243742
2000,0.138500,0.426141,0.228475
2500,0.102800,0.382843,0.209302
3000,0.069600,0.380296,0.202556


/opt/conda/lib/python3.10/site-packages/transformers/models/wav2vec2/processing_wav2vec2.py:156: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  use_reentrant(bool, optional): Use checkpointing
/opt/conda/lib/python3.10/site-packages/transformers/models/wav2vec2/processing_wav2vec2.py:156: UserWarning: `as_target_processor` is deprecated

TrainOutput(global_step=3430, training_loss=0.5063050044868848, metrics={'train_runtime': 7398.5973, 'train_samples_per_second': 14.849, 'train_steps_per_second': 0.464, 'total_flos': 3.522459452773231e+18, 'train_loss': 0.5063050044868848, 'epoch': 34.82})

In [211]:
trainer.push_to_hub()

events.out.tfevents.1713878389.6ee8ca95d332.1150.0:   0%|          | 0.00/9.89k [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/4.98k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Upload 3 LFS files:   0%|          | 0/3 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/Aviral2412/fineturning-medical_data/commit/76577e9b4fc4905d817180069777ee794c212910', commit_message='End of training', commit_description='', oid='76577e9b4fc4905d817180069777ee794c212910', pr_url=None, pr_revision=None, pr_num=None)

OSError: Can't load tokenizer for 'Aviral2412/fineturning-medical_data'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'Aviral2412/fineturning-medical_data' is the correct path to a directory containing all relevant files for a Wav2Vec2CTCTokenizer tokenizer.